# Eval: Sigma Evaluator (Detailed)

This notebook unwraps sigma evaluation into Python-level steps:
- run evaluator (optional)
- load/clean CSV tables
- inspect available metrics
- plot sigma curves inline


In [ ]:
from pathlib import Path
import argparse
import pandas as pd
import matplotlib.pyplot as plt

from eval.sigma_evaluator.run_sigma_evaluator import run_sigma_evaluator
from eval.sigma_evaluator.plot_sigma_evaluations import (
    clean_sigma_table,
    available_metrics,
    build_all_df,
    save_sigma_curves_pdf,
)


In [ ]:
# Step 1: optional run (set RUN_EVALUATOR=True for real execution)
RUN_EVALUATOR = False

cfg = dict(
    name_exp='YOUR_RUN_ID',
    name_ckpt='last.ckpt',
    ckpt_root='/home/ecm5702/scratch/aifs/checkpoint',
    out_file='sigma_eval_table.csv',
    out_csv='/tmp/sigma_eval_table_demo.csv',
    device='cpu',
    n_samples=1,
    validation_frequency='50h',
    sigmas='0.02,0.5,2.0',
    run_pure_noise=True,
    run_noised=True,
)

if RUN_EVALUATOR:
    args = argparse.Namespace(**cfg)
    out_csv = run_sigma_evaluator(args)
    print('Generated:', out_csv)
else:
    out_csv = Path(cfg['out_csv'])
    print('Skipping evaluator run. Set RUN_EVALUATOR=True to execute.')


In [ ]:
# Step 2: load one sigma table and clean
# If you already have a CSV, point to it here.
if out_csv.exists():
    df = pd.read_csv(out_csv)
    df = clean_sigma_table(df, csv_path=out_csv)
    print(df.head())
else:
    print(f'Missing CSV: {out_csv}')
    print('Provide an existing table path to continue plotting.')


In [ ]:
# Step 3: show available metric columns
if out_csv.exists():
    mets = available_metrics(df)
    print('First 25 metrics:')
    print(mets[:25])


In [ ]:
# Step 4: inline plot for a few metrics
if out_csv.exists():
    plot_metrics = ['loss']
    if 'diff_all_var_non_weighted' in df.columns:
        plot_metrics.append('diff_all_var_non_weighted')

    for metric in plot_metrics:
        plt.figure()
        for pred_flag, sub in df.groupby('prediction_on_pure_noise'):
            g = sub.sort_values('sigma')
            plt.plot(g['sigma'], g[metric], marker='o', label=f'pure_noise={pred_flag}')
        plt.xscale('log')
        plt.yscale('log')
        plt.xlabel('sigma')
        plt.ylabel(metric)
        plt.title(f'{metric} vs sigma')
        plt.grid(True)
        plt.legend()
        plt.show()


In [ ]:
# Step 5: multi-experiment workflow (optional)
# specs format: [(label, 'RUN_ID/sigma_eval_table.csv'), ...]
specs = [
    # ('expA', 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa/sigma_eval_table.csv'),
    # ('expB', 'bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb/sigma_eval_table.csv'),
]

if specs:
    all_df = build_all_df(specs)
    pdf_out = Path('/tmp/sigma_curves.pdf')
    save_sigma_curves_pdf(
        all_df=all_df,
        metrics=['loss'],
        pdf_path=pdf_out,
        sigma_min=0.0,
        pred_flags=(False, True),
    )
    print('Saved PDF:', pdf_out)
